<a href="https://colab.research.google.com/github/boluwatifeakintayo/boluwatife-FLYRANKAI-repo/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/boluwatifeakintayo/boluwatife-FLYRANKAI-repo/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from datasets import load_dataset
import pandas as pd
import itertools

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    split="train"
)

# grab first 5000 rows only, so we don't crash Colab
sample_rows = list(itertools.islice(ds, 5000))
df = pd.DataFrame(sample_rows)

print(df.shape)
print(df.columns.tolist())
df.head()
print("Rows with GSC data available:", df['gsc_data_available'].sum(), "/", len(df))
print("Rows with any impressions:", (df['gsc_impressions'] > 0).sum(), "/", len(df))

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

(5000, 30)
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']
Rows with GSC data available: 5000 / 5000
Rows with any impressions: 5000 / 5000


## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

#Lane 1: Ranking Signal Analysis
###I picked Lane 1 (Ranking Signal Analysis) because of a pattern I first noticed outside this project - exploring Amazon KDP book rankings, where I saw books rank highly without actually selling well, and other books sell well without ranking particularly high. That contradiction stuck with me: ranking position doesn't always predict real performance. I wanted to test whether the same pattern shows up in this dataset. Do web pages that rank well on Google actually get clicked, or can a page rank decently and still get ignored? That gap between "ranking" and "actual outcome" is what Lane 1 lets me dig into directly.

In [3]:
df_valid = df[df['gsc_impressions'] > 0].copy()
df_valid['ctr'] = df_valid['gsc_clicks'] / df_valid['gsc_impressions']

def position_bucket(pos):
    if pos <= 3: return "1-3 (top)"
    elif pos <= 10: return "4-10 (page 1)"
    elif pos <= 20: return "11-20 (page 2)"
    else: return "20+ (deep)"

df_valid['position_bucket'] = df_valid['gsc_avg_position'].apply(position_bucket)

summary = df_valid.groupby('position_bucket').agg(
    avg_ctr=('ctr', 'mean'),
    avg_impressions=('gsc_impressions', 'mean'),
    n_rows=('ctr', 'count')
).round(4)

print(summary)


                 avg_ctr  avg_impressions  n_rows
position_bucket                                  
1-3 (top)         0.0385          25.8424     184
11-20 (page 2)    0.0124          12.1197     919
20+ (deep)        0.0036          10.5968    2232
4-10 (page 1)     0.0161          10.7520    1665


####I found that at least on average, across position buckets ranking position and click-through rate are actually tightly linked: pages in positions 1-3 had a 3.85% average CTR, compared to just 0.36% for pages ranked 20+. This doesn't rule out my original curiosity (individual pages could still buck the trend), but it means my starting assumption doesn't hold at the aggregate level which is itself a useful, honest finding worth digging into further in this lane.




## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

#Decision
A content strategist wants to find pages that are ranking fine but still underperforming compared to similar-ranked pages because those are the ones with a hidden problem (bad title, wrong content, mismatched intent) rather than just a hard-to-rank topic.

#Who acts, and what they do

 ##### A content strategist reviews each flagged "gap page" and takes action on it — starting with rewriting the page's title tag and meta description (since these are what searchers see before clicking), and checking whether the page's content actually matches what someone searching that term is looking for

#Cost of a wrong call

##### If a page is wrongly flagged as a "gap page" (looks like it's underperforming, but actually isn't a real problem), the strategist wastes hours rewriting a title or content that didn't need fixing — time that could've gone to a genuinely broken page. If a real gap page is missed (not flagged when it should be), the business keeps losing potential clicks and traffic on a page that's already ranking well — a cheap, easy win left on the table.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [4]:
# Step 1: calculate the average CTR per bucket (we did this already, but let's merge it back)
bucket_avg = df_valid.groupby('position_bucket')['ctr'].transform('mean')

# Step 2: how far below expected is each page?
df_valid['expected_ctr'] = bucket_avg
df_valid['ctr_gap'] = df_valid['ctr'] - df_valid['expected_ctr']

# Step 3: flag pages that are ranking OK (page 1 or top) but have CTR less than half the expected average
gap_pages = df_valid[
    (df_valid['position_bucket'].isin(['1-3 (top)', '4-10 (page 1)'])) &
    (df_valid['ctr'] < df_valid['expected_ctr'] * 0.5)
]

print("Total pages ranked well (top or page 1):", len(df_valid[df_valid['position_bucket'].isin(['1-3 (top)', '4-10 (page 1)'])]))
print("Of those, 'gap pages' underperforming their expected CTR by 50%+:", len(gap_pages))
print("Percentage:", round(len(gap_pages) / len(df_valid[df_valid['position_bucket'].isin(['1-3 (top)', '4-10 (page 1)'])]) * 100, 2), "%")

Total pages ranked well (top or page 1): 1849
Of those, 'gap pages' underperforming their expected CTR by 50%+: 1615
Percentage: 87.34 %


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

##What I CAN claim:

#####In this sample of 5,000 rows (a small slice of the 78.8M-row dataset), there is a clear, consistent relationship between Google ranking position and click-through rate pages ranked 1-3 averaged a 3.85% CTR, compared to just 0.36% for pages ranked 20+. I also found that a meaningful share of well-ranked pages (top 3 or page 1) underperform their position's expected CTR by 50% or more these "gap pages" are a real, observable pattern worth investigating further, not just a rare edge case.

##What I CANNOT claim:

#####I cannot claim that ranking position causes clicks the relationship could be explained by a third factor, like strong content naturally earning both good rankings and more clicks. I also cannot claim these findings hold across the full 78.8M-row dataset, since I only sampled 5,000 rows from one time window, and results may vary by client, industry, or season. Finally, I cannot yet claim that fixing a "gap page" (e.g., rewriting its title) would actually increase its CTR that would require testing an actual intervention, not just observing existing data.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.